In [ ]:
!pip install -U lightning torchmetrics pandas scikit-learn matplotlib

import os
import re
from dataclasses import dataclass
from typing import List, Iterable, Tuple, Dict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import lightning.pytorch as pl
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor

from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score, MulticlassAUROC

SEED = 42
pl.seed_everything(SEED, workers=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

USE_CUDA = torch.cuda.is_available()
ACCELERATOR = "gpu" if USE_CUDA else "cpu"
DEVICES = 1
print("USE_CUDA =", USE_CUDA, "| ACCELERATOR =", ACCELERATOR)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 39.7 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


USE_CUDA = False | ACCELERATOR = cpu


In [ ]:
df = pd.read_csv("/content/tripadvisor_hotel_reviews.csv")
print(df.shape)
print(df.columns)
df.head()

(20491, 2)
Index(['Review', 'Rating'], dtype='object')


,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5


In [ ]:
TEXT_COL_CANDIDATES = ["Review", "review", "Text", "text"]
LABEL_COL_CANDIDATES = ["Rating", "Ratings", "rating", "ratings", "Score", "score"]

TEXT_COL = next((c for c in TEXT_COL_CANDIDATES if c in df.columns), None)
LABEL_COL = next((c for c in LABEL_COL_CANDIDATES if c in df.columns), None)

df = df[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)

print("TEXT_COL =", TEXT_COL, "| LABEL_COL =", LABEL_COL)
print(df[LABEL_COL].value_counts().sort_index())


TEXT_COL = Review | LABEL_COL = Rating
Rating
1    1421
2    1793
3    2184
4    6039
5    9054
Name: count, dtype: int64


In [ ]:
TOKEN_RE = re.compile(r"[^a-z0-9]+")

def normalize_text(s: str) -> str:
    s = str(s).lower()
    s = s.replace("<br />", " ")
    s = TOKEN_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(s: str) -> List[str]:
    return normalize_text(s).split()

y_raw = df[LABEL_COL].astype(int).values
y = y_raw - 1
texts = df[TEXT_COL].astype(str).values

X_train, X_temp, y_train, y_temp = train_test_split(
    texts, y, test_size=0.40, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("Sizes:", len(X_train), len(X_val), len(X_test))
print("Train dist:", np.bincount(y_train, minlength=5))
print("Val dist:", np.bincount(y_val, minlength=5))
print("Test dist:", np.bincount(y_test, minlength=5))


Sizes: 12294 4098 4099
Train dist: [ 853 1076 1310 3623 5432]
Val dist: [ 284  358  437 1208 1811]
Test dist: [ 284  359  437 1208 1811]


In [ ]:
from collections import Counter

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

def build_vocab(texts: Iterable[str], min_freq: int = 2, max_vocab: int = 30000) -> Dict[str, int]:
    counter = Counter()
    for s in texts:
        counter.update(tokenize(s))
    tokens = [t for t, f in counter.most_common() if f >= min_freq]
    tokens = tokens[: max(0, max_vocab - 2)]
    stoi = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
    for i, t in enumerate(tokens, start=2):
        stoi[t] = i
    return stoi

def numericalize(text: str, stoi: Dict[str,int], max_len: int):
    toks = tokenize(text)
    ids = [stoi.get(t, UNK_ID) for t in toks[:max_len]]
    length = len(ids)
    if length < max_len:
        ids += [PAD_ID] * (max_len - length)
    return torch.tensor(ids, dtype=torch.long), length

MAX_LEN = 200
VOCAB_MIN_FREQ = 2
VOCAB_MAX = 30000

stoi = build_vocab(X_train, min_freq=VOCAB_MIN_FREQ, max_vocab=VOCAB_MAX)
vocab_size = len(stoi)
print("Vocab size:", vocab_size)


Vocab size: 20407


In [ ]:
class ReviewsDataset(Dataset):
    def __init__(self, texts, labels, stoi, max_len):
        self.texts = texts
        self.labels = labels.astype(np.int64)
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x_ids, length = numericalize(self.texts[idx], self.stoi, self.max_len)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return {"input_ids": x_ids, "length": torch.tensor(length, dtype=torch.long), "label": y}

def collate_fn(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch], dim=0)
    lengths = torch.stack([b["length"] for b in batch], dim=0)
    labels = torch.stack([b["label"] for b in batch], dim=0)
    return input_ids, lengths, labels

train_ds = ReviewsDataset(X_train, y_train, stoi, MAX_LEN)
val_ds   = ReviewsDataset(X_val,   y_val,   stoi, MAX_LEN)
test_ds  = ReviewsDataset(X_test,  y_test,  stoi, MAX_LEN)

BATCH_SIZE = 128 if torch.cuda.is_available() else 64
print("BATCH_SIZE:", BATCH_SIZE)


BATCH_SIZE: 64


In [ ]:
@dataclass
class DMConfig:
    batch_size: int = BATCH_SIZE
    num_workers: int = 2

class ReviewsDataModule(pl.LightningDataModule):
    def __init__(self, train_ds, val_ds, test_ds, cfg: DMConfig):
        super().__init__()
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.test_ds = test_ds
        self.cfg = cfg

    def train_dataloader(self):
        return DataLoader(
            self.train_ds, batch_size=self.cfg.batch_size, shuffle=True,
            num_workers=self.cfg.num_workers, pin_memory=torch.cuda.is_available(),
            persistent_workers=(self.cfg.num_workers > 0), collate_fn=collate_fn
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds, batch_size=self.cfg.batch_size, shuffle=False,
            num_workers=self.cfg.num_workers, pin_memory=torch.cuda.is_available(),
            persistent_workers=(self.cfg.num_workers > 0), collate_fn=collate_fn
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_ds, batch_size=self.cfg.batch_size, shuffle=False,
            num_workers=self.cfg.num_workers, pin_memory=torch.cuda.is_available(),
            persistent_workers=(self.cfg.num_workers > 0), collate_fn=collate_fn
        )

dm = ReviewsDataModule(train_ds, val_ds, test_ds, DMConfig(num_workers=min(4, os.cpu_count() or 2)))


In [ ]:
NUM_CLASSES = 5

class BaseTextClassifier(pl.LightningModule):
    def __init__(self, lr: float = 2e-3, weight_decay: float = 1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.criterion = nn.CrossEntropyLoss()

        self.val_acc = MulticlassAccuracy(num_classes=NUM_CLASSES)
        self.val_f1  = MulticlassF1Score(num_classes=NUM_CLASSES, average="macro")
        self.val_auc = MulticlassAUROC(num_classes=NUM_CLASSES, average="macro")

        self.test_acc = MulticlassAccuracy(num_classes=NUM_CLASSES)
        self.test_f1  = MulticlassF1Score(num_classes=NUM_CLASSES, average="macro")
        self.test_auc = MulticlassAUROC(num_classes=NUM_CLASSES, average="macro")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=1, min_lr=1e-6)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sch, "monitor": "val_loss"}}

    def _shared_step(self, batch):
        input_ids, lengths, labels = batch
        logits = self(input_ids, lengths)
        loss = self.criterion(logits, labels)
        probs = torch.softmax(logits, dim=1)
        return loss, probs, labels

    def training_step(self, batch, batch_idx):
        loss, _, _ = self._shared_step(batch)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, probs, labels = self._shared_step(batch)
        self.val_acc.update(probs, labels)
        self.val_f1.update(probs, labels)
        self.val_auc.update(probs, labels)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

    def on_validation_epoch_end(self):
        self.log("val_acc", self.val_acc.compute(), prog_bar=True)
        self.log("val_f1",  self.val_f1.compute(),  prog_bar=True)
        self.log("val_auc", self.val_auc.compute(), prog_bar=True)
        self.val_acc.reset(); self.val_f1.reset(); self.val_auc.reset()

    def test_step(self, batch, batch_idx):
        loss, probs, labels = self._shared_step(batch)
        self.test_acc.update(probs, labels)
        self.test_f1.update(probs, labels)
        self.test_auc.update(probs, labels)
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

    def on_test_epoch_end(self):
        self.log("test_acc", self.test_acc.compute(), prog_bar=True)
        self.log("test_f1",  self.test_f1.compute(),  prog_bar=True)
        self.log("test_auc", self.test_auc.compute(), prog_bar=True)
        self.test_acc.reset(); self.test_f1.reset(); self.test_auc.reset()


In [ ]:
class TextCNN(BaseTextClassifier):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        num_filters: int = 128,
        kernel_sizes: Tuple[int, ...] = (3, 4, 5),
        dropout_p: float = 0.3,
        lr: float = 2e-3,
        weight_decay: float = 1e-4,
    ):
        super().__init__(lr=lr, weight_decay=weight_decay)
        self.save_hyperparameters()

        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), NUM_CLASSES)

    def forward(self, input_ids, lengths=None):
        x = self.embed(input_ids)    # (B, L, E)
        x = x.transpose(1, 2)        # (B, E, L)
        feats = []
        for conv in self.convs:
            h = torch.relu(conv(x))          # (B, F, L')
            h = torch.max(h, dim=2).values   # (B, F)
            feats.append(h)
        z = torch.cat(feats, dim=1)          # (B, F*K)
        z = self.dropout(z)
        return self.fc(z)


In [ ]:
class BiLSTM(BaseTextClassifier):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        hidden_dim: int = 128,
        num_layers: int = 1,
        dropout_p: float = 0.3,
        lr: float = 2e-3,
        weight_decay: float = 1e-4,
    ):
        super().__init__(lr=lr, weight_decay=weight_decay)
        self.save_hyperparameters()

        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=(dropout_p if num_layers > 1 else 0.0),
        )
        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Linear(hidden_dim * 2, NUM_CLASSES)

    def forward(self, input_ids, lengths):
        x = self.embed(input_ids)  # (B, L, E)
        lengths_cpu = lengths.detach().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h_fwd = h_n[-2]
        h_bwd = h_n[-1]
        h = torch.cat([h_fwd, h_bwd], dim=1)  # (B, 2H)
        h = self.dropout(h)
        return self.fc(h)


In [ ]:
class AttentionPooling(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.proj = nn.Linear(in_dim, in_dim)
        self.score = nn.Linear(in_dim, 1)

    def forward(self, h, mask):
        # h: (B, L, D), mask: (B, L) {0,1}
        u = torch.tanh(self.proj(h))
        s = self.score(u).squeeze(-1)
        s = s.masked_fill(mask == 0, -1e9)
        a = torch.softmax(s, dim=1)
        v = torch.sum(h * a.unsqueeze(-1), dim=1)
        return v

class BiLSTMAttn(BaseTextClassifier):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        hidden_dim: int = 256,
        num_layers: int = 2,
        dropout_p: float = 0.4,
        lr: float = 2e-3,
        weight_decay: float = 1e-4,
    ):
        super().__init__(lr=lr, weight_decay=weight_decay)
        self.save_hyperparameters()

        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout_p,
        )
        self.attn = AttentionPooling(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Linear(hidden_dim * 2, NUM_CLASSES)

    def forward(self, input_ids, lengths):
        x = self.embed(input_ids)
        lengths_cpu = lengths.detach().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=input_ids.size(1))

        B, L = input_ids.size(0), input_ids.size(1)
        idx = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        mask = (idx < lengths.unsqueeze(1)).long()

        v = self.attn(out, mask)
        v = self.dropout(v)
        return self.fc(v)


In [ ]:
def make_trainer(run_name: str, max_epochs: int = 10):
    tb = TensorBoardLogger(save_dir="tb_logs", name=run_name)
    csv = CSVLogger(save_dir="csv_logs", name=run_name)

    early = EarlyStopping(monitor="val_loss", mode="min", patience=2, min_delta=1e-3)
    ckpt = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1, filename="best-{epoch:02d}-{val_loss:.4f}")
    lrmon = LearningRateMonitor(logging_interval="epoch")

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator=ACCELERATOR,
        devices=DEVICES,
        deterministic=True,
        callbacks=[early, ckpt, lrmon],
        logger=[tb, csv],
        log_every_n_steps=50,
    )
    return trainer, ckpt

MAX_EPOCHS = 4

In [ ]:
results = []

# TextCNN
cnn = TextCNN(vocab_size=vocab_size)
trainer, ckpt = make_trainer("textcnn", MAX_EPOCHS)
trainer.fit(cnn, datamodule=dm)
best_cnn = TextCNN.load_from_checkpoint(ckpt.best_model_path, vocab_size=vocab_size)
cnn_test = trainer.test(best_cnn, datamodule=dm, verbose=False)[0]
results.append(("TextCNN", ckpt.best_model_path, cnn_test))

# BiLSTM
lstm = BiLSTM(vocab_size=vocab_size)
trainer, ckpt = make_trainer("bilstm", MAX_EPOCHS)
trainer.fit(lstm, datamodule=dm)
best_lstm = BiLSTM.load_from_checkpoint(ckpt.best_model_path, vocab_size=vocab_size)
lstm_test = trainer.test(best_lstm, datamodule=dm, verbose=False)[0]
results.append(("BiLSTM", ckpt.best_model_path, lstm_test))

# BiLSTM + Attn
attn = BiLSTMAttn(vocab_size=vocab_size)
trainer, ckpt = make_trainer("bilstm_attn", MAX_EPOCHS)
trainer.fit(attn, datamodule=dm)
best_attn = BiLSTMAttn.load_from_checkpoint(ckpt.best_model_path, vocab_size=vocab_size)
attn_test = trainer.test(best_attn, datamodule=dm, verbose=False)[0]
results.append(("BiLSTM+Attn", ckpt.best_model_path, attn_test))

results


Testing ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65/65 0:01:03 • 0:00:00 1.07it/s

[('TextCNN',
  'tb_logs/textcnn/version_3/checkpoints/best-epoch=02-val_loss=0.9385.ckpt',
  {'test_loss': 0.9189342260360718,
   'test_acc': 0.4592008888721466,
   'test_f1': 0.4743528962135315,
   'test_auc': 0.8614187240600586}),
 ('BiLSTM',
  'tb_logs/bilstm/version_2/checkpoints/best-epoch=01-val_loss=0.9310.ckpt',
  {'test_loss': 0.9384942650794983,
   'test_acc': 0.5176923274993896,
   'test_f1': 0.5148661136627197,
   'test_auc': 0.8519355058670044}),
 ('BiLSTM+Attn',
  'tb_logs/bilstm_attn/version_0/checkpoints/best-epoch=01-val_loss=0.8931.ckpt',
  {'test_loss': 0.884438693523407,
   'test_acc': 0.4862433075904846,
   'test_f1': 0.5100424289703369,
   'test_auc': 0.8716743588447571})]

In [ ]:
rows = []
for name, ckpt_path, m in results:
    rows.append({
        "model": name,
        "best_ckpt": ckpt_path,
        "test_loss": float(m.get("test_loss", np.nan)),
        "test_acc": float(m.get("test_acc", np.nan)),
        "test_f1": float(m.get("test_f1", np.nan)),
        "test_auc": float(m.get("test_auc", np.nan)),
    })

cmp_df = pd.DataFrame(rows).sort_values("test_f1", ascending=False)
cmp_df


In [ ]:
import glob
import matplotlib.pyplot as plt

def load_metrics_csv(run_name: str) -> pd.DataFrame:
    base = os.path.join("csv_logs", run_name)
    versions = sorted(glob.glob(os.path.join(base, "version_*")))
    if not versions:
        raise FileNotFoundError(f"Не найдены логи для {run_name} в {base}")
    return pd.read_csv(os.path.join(versions[-1], "metrics.csv"))

def plot_metric(run_name: str, metric: str):
    m = load_metrics_csv(run_name)
    epoch_m = m[m["step"].isna()] if "step" in m.columns else m
    if metric not in epoch_m.columns:
        print(f"[{run_name}] нет {metric}. Доступно: {list(epoch_m.columns)}")
        return
    plt.figure()
    plt.plot(epoch_m["epoch"], epoch_m[metric])
    plt.xlabel("epoch"); plt.ylabel(metric); plt.title(f"{run_name}: {metric}")
    plt.show()

for run in ["textcnn", "bilstm", "bilstm_attn"]:
    for metric in ["train_loss", "val_loss", "val_f1", "val_auc"]:
        plot_metric(run, metric)


In [ ]:
best_name = cmp_df.iloc[0]["model"]
print("Best by test_f1:", best_name)

if best_name == "TextCNN":
    best_model = best_cnn
elif best_name == "BiLSTM":
    best_model = best_lstm
else:
    best_model = best_attn

device = "cuda" if torch.cuda.is_available() else "cpu"
best_model = best_model.to(device).eval()

all_probs, all_y = [], []
with torch.no_grad():
    for input_ids, lengths, yb in dm.test_dataloader():
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)
        logits = best_model(input_ids, lengths)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_y.append(yb.numpy())

all_probs = np.concatenate(all_probs, axis=0)
all_y = np.concatenate(all_y, axis=0)
y_pred = all_probs.argmax(axis=1)

print(classification_report(all_y, y_pred, digits=4))
